<a href="https://colab.research.google.com/github/sr606/LLM/blob/main/Mermaid6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#graph_model_replace


class Graph:
    def __init__(self):
        self.stages = {}          # stage_id -> stage object
        self.datasets = set()     # dataset names
        self.stage_edges = []     # (dataset → stage) and (stage → dataset)
        self.column_edges = []    # (source_column, target_column, expression)

    def add_stage(self, stage_id):
        self.stages[stage_id] = {}

    def add_dataset(self, dataset_name):
        self.datasets.add(dataset_name)

    def add_stage_edge(self, source, target):
        self.stage_edges.append((source, target))

    def add_column_edge(self, src_col, tgt_col, expression):
        self.column_edges.append((src_col, tgt_col, expression))



#only build_graph() function
    def build_graph(lineage_response):
    graph = Graph()

    for stage in lineage_response.stages:
        graph.add_stage(stage.stage_id)

        # Inputs → Stage
        for inp in stage.inputs:
            dataset = inp["dataset_name"]
            graph.add_dataset(dataset)
            graph.add_stage_edge(dataset, stage.stage_id)

        # Stage → Outputs
        for out in stage.outputs:
            dataset = out["dataset_name"]
            graph.add_dataset(dataset)
            graph.add_stage_edge(stage.stage_id, dataset)

        # Column lineage
        for col in stage.column_lineage:
            for src in col.source_columns:
                graph.add_column_edge(
                    src,
                    f"{stage.stage_id}.{col.target_column}",
                    col.expression
                )

    return graph



#renderer new
from graphviz import Digraph


def render_pipeline_view(graph, output_path):
    dot = Digraph(format="pdf")
    dot.attr(rankdir="LR")

    # Dataset nodes
    for dataset in graph.datasets:
        dot.node(dataset, shape="box", style="filled", fillcolor="#E3F2FD")

    # Stage nodes
    for stage in graph.stages:
        dot.node(stage, shape="ellipse", style="filled", fillcolor="#FFF3E0")

    # Edges
    for src, tgt in graph.stage_edges:
        dot.edge(src, tgt)

    dot.render(output_path, cleanup=True)



def render_stage_column_view(graph, stage_id, output_path):
    dot = Digraph(format="pdf")
    dot.attr(rankdir="LR")

    dot.node(stage_id, shape="ellipse", style="filled", fillcolor="#FFE0B2")

    for src, tgt, expr in graph.column_edges:
        if tgt.startswith(stage_id + "."):
            dot.node(src, shape="box", fillcolor="#E8F5E9", style="filled")
            dot.node(tgt, shape="box", fillcolor="#FFF9C4", style="filled")

            label = expr[:40] + "..." if len(expr) > 40 else expr
            dot.edge(src, tgt, label=label)

    dot.render(output_path, cleanup=True)